# Notebook 3: NLP Recommendation Pipeline & Production Delivery
**Objective:** Bypass the deprecated Spotify Audio Features API by engineering a Natural Language Processing (NLP) architecture. We will translate crowd-sourced human tags (via Last.fm) into mathematical vectors using **TF-IDF**, calculate **Cosine Similarity** against our local database, and push the final diverse playlist directly to a live Spotify account via the Spotify Web API.

In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import spotipy
from spotipy.oauth2 import SpotifyOAuth

# 0. Load the secret variables from the hidden .env file
load_dotenv()

# 1. Load the clean dataset
df_clean = pd.read_csv('data/cleaned_spotify_tracks.csv')

# Fill empty genres with a blank string to prevent NLP math errors
df_clean['track_genre'] = df_clean['track_genre'].fillna("")

# 2. Initialize the NLP Vectorizer
print("Training the TF-IDF Vectorizer on 113,000 tracks...")
tfidf = TfidfVectorizer(stop_words='english')

# Fit the model to learn our entire database's vocabulary
tfidf_matrix = tfidf.fit_transform(df_clean['track_genre'])

print("NLP Pipeline Ready!\n")

Training the TF-IDF Vectorizer on 113,000 tracks...
NLP Pipeline Ready!



In [ ]:
import requests
import os

def get_live_song_tags(track_name, artist_name, api_key):
    url = "http://ws.audioscrobbler.com/2.0/"
    
    params = {
        "method": "track.gettoptags",
        "track": track_name,
        "artist": artist_name,
        "api_key": api_key,
        "format": "json"
    }
    
    response = requests.get(url, params=params)
    data = response.json()
    
    # --- DEBUGGING SECTION ---
    if "error" in data:
        print(f"🚨 LAST.FM API ERROR: {data['message']}")
        return ""
    if "toptags" not in data or not data["toptags"]["tag"]:
        print("🚨 NO TAGS FOUND: The song might be too obscure or spelled wrong.")
        return ""
    # ------------------------------
        
    tags = [tag["name"] for tag in data["toptags"]["tag"][:5]]
    return " ".join(tags)

LASTFM_API_KEY = os.getenv("LASTFM_API_KEY")
live_user_tags = get_live_song_tags("Take on Me", "a-ha", LASTFM_API_KEY)
print("Fetched Tags:", live_user_tags)

Fetched Tags: 80s pop new wave synthpop synth pop


### 1. Generating the Target Vibe (API Mock)
*💡 **Note for Reviewers:** To ensure this notebook runs instantly without requiring you to input external API keys, the cell below uses a pre-fetched (mocked) Last.fm tag response. In production, this text string is dynamically generated by sending the user's input tracks to the `http://ws.audioscrobbler.com/2.0/` endpoint.*

In [3]:
# MOCKED LAST.FM RESPONSE for a high-energy, upbeat pop vibe
live_user_tags = "synthpop electronic dance pop 80s upbeat synth"
print(f"Target NLP Vibe: {live_user_tags}")

# 1. Translate the human words into a math vector
target_vector = tfidf.transform([live_user_tags])

# 2. Calculate distances against the entire database
similarities = cosine_similarity(target_vector, tfidf_matrix)[0]
df_clean['nlp_match_score'] = similarities

# 3. Grab the top 20 mathematically closest tracks, dropping clones
top_matches = df_clean.sort_values(by=['nlp_match_score', 'popularity'], ascending=[False, False])
final_playlist = top_matches.drop_duplicates(subset=['track_name', 'artists']).head(20)

print("\n🎧 TOP 5 CANDIDATES FOUND:")
print(final_playlist[['track_name', 'artists', 'nlp_match_score']].head())

Target NLP Vibe: synthpop electronic dance pop 80s upbeat synth

🎧 TOP 5 CANDIDATES FOUND:
                                          track_name  \
107000             Everybody Wants To Rule The World   
107009                                    Take on Me   
107007  Sweet Dreams (Are Made of This) - Remastered   
107008                   Wake Me Up Before You Go-Go   
107015                       Never Gonna Give You Up   

                                     artists  nlp_match_score  
107000                       Tears For Fears         0.669042  
107009                                  a-ha         0.669042  
107007  Eurythmics;Annie Lennox;Dave Stewart         0.669042  
107008                                 Wham!         0.669042  
107015                           Rick Astley         0.669042  


### 2. Pushing to Production (Spotify API Delivery)
Now that the recommendation pipeline has computed the top 20 nearest neighbors, we authenticate with the Spotify API to automatically construct a public playlist on the user's account. This demonstrates a complete, multi-API data flow from local processing to a live product feature. 

**Work in Progress**

In [12]:
# IMPORTANT: Use your newly rotated credentials here!
CLIENT_ID = 'your_new_client_id'
CLIENT_SECRET = 'your_new_client_secret'

# Authenticate with permission to build public playlists
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    redirect_uri="http://localhost:9000",
    scope="playlist-modify-public"
))

# 1. Format the Spotify Track URIs
# (Check if your dataset uses 'track_id' or 'id' as the column name!)
track_uris = ["spotify:track:" + str(tid) for tid in final_playlist['track_id'].tolist()]

# 2. Get your personal user ID
user_id = sp.current_user()['id']

# 3. Create the empty playlist
playlist_name = "My ML Generated Vibe"
new_playlist = sp.user_playlist_create(
    user=user_id, 
    name=playlist_name, 
    public=True, 
    description="Generated via TF-IDF & Cosine Similarity in Python!"
)

# 4. Push the tracks to the playlist
sp.playlist_add_items(playlist_id=new_playlist['id'], items=track_uris)

print(f"✅ Success! Playlist created.")
print(f"🔗 Check it out here: {new_playlist['external_urls']['spotify']}")

Using 'localhost' as a redirect URI is being deprecated. Use a loopback IP address such as 127.0.0.1 to ensure your app remains functional.


PermissionError: [WinError 10013] An attempt was made to access a socket in a way forbidden by its access permissions